## Tổng quan dữ liệu

- 2930 giao dịch bán nhà ở Ames, Iowa (2006-2010)
- 80 biến: 20 liên tục, 14 rời rạc, 23 định danh, 23 thứ tự
- Biến mục tiêu: `SalePrice`
- `PID` chỉ là mã thửa đất, không có ý nghĩa dự báo nên bỏ khi phân tích

## import thư viện

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [3]:
df = pd.read_csv(r'data\train.csv')



Dữ liệu là giao dịch bán nhà ở Ames, Iowa (2006-2010), theo paper của De Cock (2011) có 2930 giao dịch và 80 biến. File đang dùng là `train.csv` (bản Kaggle House Prices, chia từ cùng bộ dữ liệu này), nên chỉ có:

- 1460 dòng, 81 cột (gồm `Id` và biến mục tiêu `SalePrice`)
- 38 cột số, 43 cột chữ
- `Id` chỉ là số thứ tự, không dùng để dự báo

In [5]:
df.shape

(1460, 81)

In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   str    
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   str    
 6   Alley          91 non-null     str    
 7   LotShape       1460 non-null   str    
 8   LandContour    1460 non-null   str    
 9   Utilities      1460 non-null   str    
 10  LotConfig      1460 non-null   str    
 11  LandSlope      1460 non-null   str    
 12  Neighborhood   1460 non-null   str    
 13  Condition1     1460 non-null   str    
 14  Condition2     1460 non-null   str    
 15  BldgType       1460 non-null   str    
 16  HouseStyle     1460 non-null   str    
 17  OverallQual    1460 non-null   int64  
 18  OverallCond    1460

### Phân loại biến

- **Liên tục:** chủ yếu là các loại diện tích (lô đất, tầng hầm, diện tích sinh hoạt, hiên...)
- **Rời rạc:** số phòng ngủ, phòng tắm, bếp, sức chứa gara, năm xây/sửa
- **Định danh:** loại nhà, vật liệu, khu vực... trong đó `Neighborhood` có 28 mức, nhiều nhất
- **Thứ tự:** các thang chất lượng (chất lượng tổng thể, bếp, tầng hầm...)

In [6]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Id,1460.0,730.500000,421.610009,1.0,365.75,730.5,1095.25,1460.0
MSSubClass,1460.0,56.897260,42.300571,20.0,20.00,50.0,70.00,190.0
LotFrontage,1201.0,70.049958,24.284752,21.0,59.00,69.0,80.00,313.0
LotArea,1460.0,10516.828082,9981.264932,1300.0,7553.50,9478.5,11601.50,215245.0
OverallQual,1460.0,6.099315,1.382997,1.0,5.00,6.0,7.00,10.0
OverallCond,1460.0,5.575342,1.112799,1.0,5.00,5.0,6.00,9.0
YearBuilt,1460.0,1971.267808,30.202904,1872.0,1954.00,1973.0,2000.00,2010.0
YearRemodAdd,1460.0,1984.865753,20.645407,1950.0,1967.00,1994.0,2004.00,2010.0
MasVnrArea,1452.0,103.685262,181.066207,0.0,0.00,0.0,166.00,1600.0
BsmtFinSF1,1460.0,443.639726,456.098091,0.0,0.00,383.5,712.25,5644.0


In [7]:
df.select_dtypes(exclude='number').describe().T

,count,unique,top,freq
MSZoning,1460,5,RL,1151
Street,1460,2,Pave,1454
Alley,91,2,Grvl,50
LotShape,1460,4,Reg,925
LandContour,1460,4,Lvl,1311
Utilities,1460,2,AllPub,1459
LotConfig,1460,5,Inside,1052
LandSlope,1460,3,Gtl,1382
Neighborhood,1460,25,NAmes,225
Condition1,1460,9,Norm,1260


### Nhận xét ban đầu

- Dữ liệu 1460 x 81, nhiều cột thiếu (19 cột), nhiều cột bị thiếu đến 99% dữ liệu phải xem xét lại Dataset liệu có phải missing thật hay là thuộc tính của căn nhà là không có (không sở hữu) những đặc trưng này


In [9]:
skew = df.select_dtypes('number').skew().sort_values(ascending=False)
print("SalePrice skew:", df['SalePrice'].skew())
skew.head(10)

SalePrice skew: 1.8828757597682129


MiscVal          24.476794
PoolArea         14.828374
LotArea          12.207688
3SsnPorch        10.304342
LowQualFinSF      9.011341
KitchenAbvGr      4.488397
BsmtFinSF2        4.255261
ScreenPorch       4.122214
BsmtHalfBath      4.103403
EnclosedPorch     3.089872
dtype: float64

- `SalePrice` lệch phải (skew = 1), trung bình lớn hơn trung vị
- Các cột lệch mạnh nhất: MiscVal  (nhiều cột diện tích có phần lớn giá trị bằng 0, như `PoolArea`, `MiscVal`, `3SsnPorch`)

### Ghi chú từ paper

- Tác giả khuyên loại nhà có `GrLivArea` > 4000 sq ft (5 điểm bất thường)
- Biến thứ tự và biến rời rạc có thể cho hệ số không đơn điệu khi mã hóa thành biến giả
- Hai ý này để dành cho phần outliers và phân tích đa biến